In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
import os
from pathlib import Path
from tqdm.auto import tqdm


FINAL_PATH = "culture_outcome_rl_format.csv"
RAW_PATH = "culture_markets_2024_to_now_closed_raw.csv"

OUTPUT_PATH = "culture_outcome_rl_format_with_polymarket_random_date.csv"
CACHE_PATH = "polymarket_market_cache.json"
PROGRESS_PATH = "culture_outcome_rl_format_with_polymarket_random_date_PROGRESS.csv"

RANDOM_SEED = 42

GAMMA_HOST = "https://gamma-api.polymarket.com"
CLOB_HOST = "https://clob.polymarket.com"


# вспомогательные функции
def safe_json_loads(x):
    if pd.isna(x):
        return None

    if isinstance(x, (list, dict)):
        return x

    try:
        return json.loads(x)
    except Exception:
        return None


def to_utc_datetime(x):
    return pd.to_datetime(x, errors="coerce", utc=True)


def normalize_date_for_merge(x):
    dt = to_utc_datetime(x)
    if pd.isna(dt):
        return None
    return dt.date().isoformat()


def load_cache(path):
    path = Path(path)

    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

# получаем полную инфу о рынке, там нужен clobTokenIds
def fetch_gamma_market_by_id(market_id, cache, sleep_sec=0.1):
    market_id = str(int(float(market_id)))

    if market_id in cache:
        return cache[market_id]

    url = f"{GAMMA_HOST}/markets/{market_id}"

    try:
        r = requests.get(url, timeout=30)

        if r.status_code != 200:
            cache[market_id] = {
                "_error": f"status_code={r.status_code}",
                "_text": r.text[:500],
            }
            return cache[market_id]

        data = r.json()
        cache[market_id] = data

        time.sleep(sleep_sec)
        return data

    except Exception as e:
        cache[market_id] = {
            "_error": str(e)
        }
        return cache[market_id]

# конкретно YES токен
def extract_yes_token_id(gamma_market):
    if not isinstance(gamma_market, dict):
        return np.nan

    if "_error" in gamma_market:
        return np.nan

    outcomes = safe_json_loads(gamma_market.get("outcomes"))
    clob_token_ids = safe_json_loads(gamma_market.get("clobTokenIds"))

    if not isinstance(outcomes, list) or not isinstance(clob_token_ids, list):
        return np.nan

    outcomes_lower = [str(x).strip().lower() for x in outcomes]

    if "yes" not in outcomes_lower:
        return np.nan

    yes_idx = outcomes_lower.index("yes")

    if yes_idx >= len(clob_token_ids):
        return np.nan

    return str(clob_token_ids[yes_idx])

# равномерно выбираем timestamp между start_dt и end_dt.
def sample_uniform_timestamp(start_dt, end_dt, rng):
    start_dt = to_utc_datetime(start_dt)
    end_dt = to_utc_datetime(end_dt)

    if pd.isna(start_dt) or pd.isna(end_dt):
        return pd.NaT

    if end_dt <= start_dt:
        return pd.NaT

    start_ts = int(start_dt.timestamp())
    end_ts = int(end_dt.timestamp())

    sampled_ts = rng.integers(start_ts, end_ts + 1)

    return pd.to_datetime(sampled_ts, unit="s", utc=True)


def choose_market_end(row):
    closed_time = to_utc_datetime(row.get("closedTime"))
    end_date = to_utc_datetime(row.get("endDate"))

    if not pd.isna(closed_time):
        return closed_time

    return end_date

# подбираем исторические цены
def fetch_price_history(token_id, start_ts, end_ts, fidelity=60):
    url = f"{CLOB_HOST}/prices-history"

    params = {
        "market": str(token_id),
        "startTs": int(start_ts),
        "endTs": int(end_ts),
        "fidelity": int(fidelity),
    }

    try:
        r = requests.get(url, params=params, timeout=30)

        if r.status_code != 200:
            return []

        data = r.json()
        return data.get("history", [])

    except Exception:
        return []

# берем цену, ближайшую к выбранной дате, если нужно постепенно расширяем окно
def get_price_near_date(token_id, target_dt, windows_hours=(12, 24, 72, 168), fidelity=60):
    if pd.isna(token_id) or pd.isna(target_dt):
        return np.nan

    target_dt = to_utc_datetime(target_dt)

    if pd.isna(target_dt):
        return np.nan

    center_ts = int(target_dt.timestamp())

    for window in windows_hours:
        start_ts = center_ts - window * 3600
        end_ts = center_ts + window * 3600

        history = fetch_price_history(
            token_id=token_id,
            start_ts=start_ts,
            end_ts=end_ts,
            fidelity=fidelity
        )

        if not history:
            continue

        hist_df = pd.DataFrame(history)

        if "t" not in hist_df.columns or "p" not in hist_df.columns:
            continue

        hist_df["dist"] = (hist_df["t"] - center_ts).abs()
        best = hist_df.sort_values("dist").iloc[0]

        return float(best["p"])

    return np.nan


# загрузка данных
final_df = pd.read_csv(FINAL_PATH)
raw_df = pd.read_csv(RAW_PATH, low_memory=False)

print("final_df:", final_df.shape)
print("raw_df:", raw_df.shape)

final_df["merge_close_date"] = final_df["question_close_date"].apply(normalize_date_for_merge)
raw_df["merge_close_date"] = raw_df["endDate"].apply(normalize_date_for_merge)

# еще отбор дубликатов
raw_df["volumeNum_numeric"] = pd.to_numeric(raw_df.get("volumeNum"), errors="coerce")

raw_for_merge = raw_df.copy()

raw_for_merge = raw_for_merge.sort_values(
    by=["question", "merge_close_date", "volumeNum_numeric"],
    ascending=[True, True, False]
)

raw_for_merge = raw_for_merge.drop_duplicates(
    subset=["question", "merge_close_date"],
    keep="first"
)

cols_from_raw = [
    "market_id",
    "condition_id",
    "question",
    "slug",
    "startDate",
    "endDate",
    "closedTime",
    "volume",
    "volumeNum",
    "liquidity",
    "liquidityNum",
    "outcomes_raw",
    "outcomePrices_raw",
    "lastTradePrice",
    "merge_close_date",
]

cols_from_raw = [c for c in cols_from_raw if c in raw_for_merge.columns]

merged = final_df.merge(
    raw_for_merge[cols_from_raw],
    left_on=["question_text", "merge_close_date"],
    right_on=["question", "merge_close_date"],
    how="left"
)

print("merged:", merged.shape)
print("market_id missing share:", merged["market_id"].isna().mean())

# volume можно сразу заполнить из raw
if "volumeNum" in merged.columns:
    merged["volume"] = pd.to_numeric(merged["volumeNum"], errors="coerce")
elif "volume_y" in merged.columns:
    merged["volume"] = pd.to_numeric(merged["volume_y"], errors="coerce")


# рандомная дата предсказания
rng = np.random.default_rng(RANDOM_SEED)

sampled_dates = []

for _, row in merged.iterrows():
    start_dt = row.get("startDate")
    end_dt = choose_market_end(row)

    sampled_dt = sample_uniform_timestamp(
        start_dt=start_dt,
        end_dt=end_dt,
        rng=rng
    )

    sampled_dates.append(sampled_dt)

merged["sampled_prediction_date"] = sampled_dates

print("sampled date missing share:", merged["sampled_prediction_date"].isna().mean())

cache = load_cache(CACHE_PATH)

yes_token_ids = []

for i, row in tqdm(merged.iterrows(), total=len(merged), desc="Fetching YES token ids"):
    market_id = row.get("market_id")

    if pd.isna(market_id):
        yes_token_ids.append(np.nan)
        continue

    gamma_market = fetch_gamma_market_by_id(
        market_id=market_id,
        cache=cache,
        sleep_sec=0.05
    )

    yes_token_id = extract_yes_token_id(gamma_market)
    yes_token_ids.append(yes_token_id)

    if i % 200 == 0:
        save_cache(cache, CACHE_PATH)

save_cache(cache, CACHE_PATH)

merged["yes_token_id"] = yes_token_ids

print("yes_token_id missing share:", merged["yes_token_id"].isna().mean())


# соотносит историческую цену с полимаркета
prices = []

start_i = 0

# если уже есть progress-файл, продолжим с него
if os.path.exists(PROGRESS_PATH):
    progress_df = pd.read_csv(PROGRESS_PATH, low_memory=False)

    if "prediction_polymarket" in progress_df.columns:
        merged = progress_df.copy()
        prices = merged["prediction_polymarket"].tolist()

        done_mask = merged["prediction_polymarket"].notna()
        start_i = int(done_mask.sum())

        print(f"Loaded progress. Already filled: {start_i}/{len(merged)}")
else:
    merged["prediction_polymarket"] = np.nan


for i in tqdm(range(len(merged)), desc="Fetching historical prices"):
    if pd.notna(merged.loc[i, "prediction_polymarket"]):
        continue

    token_id = merged.loc[i, "yes_token_id"]
    target_dt = merged.loc[i, "sampled_prediction_date"]

    price = get_price_near_date(
        token_id=token_id,
        target_dt=target_dt,
        windows_hours=(12, 24, 72, 168),
        fidelity=60
    )

    merged.loc[i, "prediction_polymarket"] = price

    if i % 50 == 0:
        merged.to_csv(PROGRESS_PATH, index=False)

    time.sleep(0.10)

merged.to_csv(PROGRESS_PATH, index=False)

# сохраняем полезные технические поля
technical_cols = [
    "market_id",
    "condition_id",
    "slug",
    "yes_token_id",
    "sampled_prediction_date",
    "startDate",
    "endDate",
    "closedTime",
    "liquidity",
    "liquidityNum",
    "lastTradePrice",
]

# убираем лишние колонки
drop_cols = [
    "question",
    "merge_close_date",
    "outcomes_raw",
    "outcomePrices_raw",
    "volumeNum",
    "volumeNum_numeric",
]

merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns], errors="ignore")

# базовый порядок
base_cols = [
    "question_id",
    "question_text",
    "question_close_date",
    "volume",
    "resolution",
    "prompt",
    "prediction_polymarket",
    "prediction_o1",
    "prediction_remax",
    "reasoning_remax",
]

existing_base_cols = [c for c in base_cols if c in merged.columns]
existing_technical_cols = [c for c in technical_cols if c in merged.columns]
other_cols = [c for c in merged.columns if c not in existing_base_cols + existing_technical_cols]

merged = merged[existing_base_cols + existing_technical_cols + other_cols]

merged.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("shape:", merged.shape)
print("prediction_polymarket missing share:", merged["prediction_polymarket"].isna().mean())
print("prediction_polymarket summary:")
print(merged["prediction_polymarket"].describe())